In [ ]:
!pip install -q timm

In [ ]:
def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

import os
import json
import copy
import random
import zipfile
import hashlib
import warnings
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm.notebook import tqdm

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import timm

warnings.filterwarnings("ignore")
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

SEED = 42
set_all_seeds(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Mounted at /content/gdrive
Using device: cuda


In [ ]:
import os
import zipfile

zip_file_path = '/content/gdrive/MyDrive/archive (2).zip'
extract_dir = '/content/unzipped_data'

# Check if the file exists before attempting to unzip
if os.path.exists(zip_file_path):
    os.makedirs(extract_dir, exist_ok=True)

    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

    print(f"Dataset extracted to: {extract_dir}")
else:
    print(f"Error: File not found at {zip_file_path}")
    print("Please ensure Google Drive is mounted and the file path is correct.")

Dataset extracted to: /content/unzipped_data


In [ ]:
base_data_dir = '/content/unzipped_data/Data'
image_extensions = ('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff')

def find_images_and_labels_from_subdirs(root_dir, extensions):
    image_paths = []
    labels = []

    for class_name in os.listdir(root_dir):
        class_dir = os.path.join(root_dir, class_name)

        if os.path.isdir(class_dir):
            for file in os.listdir(class_dir):
                if file.lower().endswith(extensions):
                    image_paths.append(os.path.join(class_dir, file))
                    labels.append(class_name)

    return image_paths, labels

train_paths, train_labels = find_images_and_labels_from_subdirs(
    os.path.join(base_data_dir, 'train'),
    image_extensions
)

valid_paths, valid_labels = find_images_and_labels_from_subdirs(
    os.path.join(base_data_dir, 'valid'),
    image_extensions
)

test_paths, test_labels = find_images_and_labels_from_subdirs(
    os.path.join(base_data_dir, 'test'),
    image_extensions
)

train_initial_df = pd.DataFrame({
    'filepath': train_paths,
    'label': train_labels,
    'original_split': 'train'
})

valid_initial_df = pd.DataFrame({
    'filepath': valid_paths,
    'label': valid_labels,
    'original_split': 'valid'
})

test_initial_df = pd.DataFrame({
    'filepath': test_paths,
    'label': test_labels,
    'original_split': 'test'
})

print("Original train:", len(train_initial_df))
print("Original valid:", len(valid_initial_df))
print("Original test :", len(test_initial_df))

Original train: 613
Original valid: 72
Original test : 315


In [ ]:
label_mapping = {
    'normal':                                            'normal',
    'adenocarcinoma_left.lower.lobe_T2_N0_M0_Ib':       'adenocarcinoma',
    'adenocarcinoma':                                    'adenocarcinoma',
    'squamous.cell.carcinoma_left.hilum_T1_N2_M0_IIIa': 'squamous.cell.carcinoma',
    'squamous.cell.carcinoma':                          'squamous.cell.carcinoma',
    'large.cell.carcinoma_left.hilum_T2_N2_M0_IIIa':    'large.cell.carcinoma',
    'large.cell.carcinoma':                             'large.cell.carcinoma',
}

all_df = pd.concat(
    [train_initial_df, valid_initial_df, test_initial_df],
    ignore_index=True
)

all_df['label'] = all_df['label'].map(label_mapping)
all_df = all_df.dropna(subset=['label']).reset_index(drop=True)

print("Total images before duplicate removal:", len(all_df))
print("\nClass distribution before duplicate removal:")
print(all_df['label'].value_counts())

Total images before duplicate removal: 1000

Class distribution before duplicate removal:
label
adenocarcinoma             338
squamous.cell.carcinoma    260
normal                     215
large.cell.carcinoma       187
Name: count, dtype: int64


In [ ]:
def file_hash(path):
    h = hashlib.md5()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)

    return h.hexdigest()

all_df['file_hash'] = all_df['filepath'].apply(file_hash)

print("Total rows:", len(all_df))
print("Unique hashes:", all_df['file_hash'].nunique())
print("Exact duplicate rows:", len(all_df) - all_df['file_hash'].nunique())

Total rows: 1000
Unique hashes: 847
Exact duplicate rows: 153


In [ ]:
conflict_hashes = (
    all_df.groupby('file_hash')['label']
    .nunique()
    .reset_index()
)

conflict_hashes = conflict_hashes[conflict_hashes['label'] > 1]

print("Conflicting duplicate hashes:", len(conflict_hashes))

if len(conflict_hashes) > 0:
    conflict_examples = all_df[all_df['file_hash'].isin(conflict_hashes['file_hash'])]
    display(conflict_examples.sort_values('file_hash').head(50))

Conflicting duplicate hashes: 1


,filepath,label,original_split,file_hash
43,/content/unzipped_data/Data/train/large.cell.c...,large.cell.carcinoma,train,acd405d5bb971e46887c9ebc14e2f48f
536,/content/unzipped_data/Data/train/squamous.cel...,squamous.cell.carcinoma,train,acd405d5bb971e46887c9ebc14e2f48f


In [ ]:
clean_df = all_df.drop_duplicates(subset=['file_hash']).reset_index(drop=True)

print("Total images after duplicate removal:", len(clean_df))
print("Removed duplicates:", len(all_df) - len(clean_df))

print("\nClean class distribution:")
print(clean_df['label'].value_counts())

Total images after duplicate removal: 847
Removed duplicates: 153

Clean class distribution:
label
adenocarcinoma             337
squamous.cell.carcinoma    257
large.cell.carcinoma       187
normal                      66
Name: count, dtype: int64


In [ ]:
import shutil
import os
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

# 1. Configuration and Setup
local_drive_output_path = '/content/gdrive/MyDrive/Cleaned_Dataset_847_Split'
SEED = 42

# 2. Creating the splits directly to avoid NameError
# First, split into (Train+Val) and Test
train_val_df, test_df = train_test_split(
    clean_df,
    test_size=0.20,
    stratify=clean_df['label'],
    random_state=SEED
)

# Second, split (Train+Val) into separate Train and Val sets
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.20,
    stratify=train_val_df['label'],
    random_state=SEED
)

# Define the mapping for iteration
splits = {
    'train': train_df,
    'val': val_df,
    'test': test_df
}

print(f"Organizing 847 images into splits at: {local_drive_output_path}")

# 3. Iterate through each split and copy files
for split_name, df in splits.items():
    print(f"\nProcessing {split_name} split ({len(df)} images)...")
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Copying {split_name}"):
        label = row['label']
        src_path = row['filepath']

        # Create path: base/split/label
        target_dir = os.path.join(local_drive_output_path, split_name, label)
        os.makedirs(target_dir, exist_ok=True)

        # Copy file keeping metadata
        filename = os.path.basename(src_path)
        dst_path = os.path.join(target_dir, filename)

        if os.path.exists(src_path):
            shutil.copy2(src_path, dst_path)
        else:
            print(f"Warning: Source not found: {src_path}")

print("\nDone! Your dataset is now organized by Split and Class in Google Drive.")

Organizing 847 images into splits at: /content/gdrive/MyDrive/Cleaned_Dataset_847_Split

Processing train split (541 images)...


Copying train:   0%|          | 0/541 [00:00<?, ?it/s]


Processing val split (136 images)...


Copying val:   0%|          | 0/136 [00:00<?, ?it/s]


Processing test split (170 images)...


Copying test:   0%|          | 0/170 [00:00<?, ?it/s]


Done! Your dataset is now organized by Split and Class in Google Drive.


split

In [ ]:
clean_train_val_df, clean_test_df = train_test_split(
    clean_df,
    test_size=0.20,
    stratify=clean_df['label'],
    random_state=SEED
)

clean_train_val_df = clean_train_val_df.reset_index(drop=True)
clean_test_df = clean_test_df.reset_index(drop=True)

print("Clean Train+Val:", len(clean_train_val_df))
print("Clean Final Test:", len(clean_test_df))

print("\nClean Train+Val distribution:")
print(clean_train_val_df['label'].value_counts())

print("\nClean Final Test distribution:")
print(clean_test_df['label'].value_counts())

Clean Train+Val: 677
Clean Final Test: 170

Clean Train+Val distribution:
label
adenocarcinoma             269
squamous.cell.carcinoma    206
large.cell.carcinoma       149
normal                      53
Name: count, dtype: int64

Clean Final Test distribution:
label
adenocarcinoma             68
squamous.cell.carcinoma    52
large.cell.carcinoma       37
normal                     13
Name: count, dtype: int64


duplivcate hasshes

In [ ]:
def check_hash_overlap(df1, df2, name1, name2):
    overlap = set(df1['file_hash']) & set(df2['file_hash'])
    print(f"{name1} vs {name2}: {len(overlap)} duplicate hashes")

check_hash_overlap(clean_train_val_df, clean_test_df, "Clean TrainVal", "Clean Final Test")

Clean TrainVal vs Clean Final Test: 0 duplicate hashes


In [ ]:
gdrive_base_output_dir = '/content/gdrive/MyDrive/DL_Model_Outputs_CLEAN_NO_LEAKAGE'
os.makedirs(gdrive_base_output_dir, exist_ok=True)

clean_train_val_csv = os.path.join(gdrive_base_output_dir, "clean_train_val_split.csv")
clean_test_csv = os.path.join(gdrive_base_output_dir, "clean_final_test_split.csv")

clean_train_val_df.to_csv(clean_train_val_csv, index=False)
clean_test_df.to_csv(clean_test_csv, index=False)

print("Saved:", clean_train_val_csv)
print("Saved:", clean_test_csv)

Saved: /content/gdrive/MyDrive/DL_Model_Outputs_CLEAN_NO_LEAKAGE/clean_train_val_split.csv
Saved: /content/gdrive/MyDrive/DL_Model_Outputs_CLEAN_NO_LEAKAGE/clean_final_test_split.csv


class names

In [ ]:
class_names = sorted(clean_df['label'].unique().tolist())
num_classes = len(class_names)
labels_map = {label: i for i, label in enumerate(class_names)}

print("Class names:", class_names)
print("Num classes:", num_classes)
print("Labels map:", labels_map)

Class names: ['adenocarcinoma', 'large.cell.carcinoma', 'normal', 'squamous.cell.carcinoma']
Num classes: 4
Labels map: {'adenocarcinoma': 0, 'large.cell.carcinoma': 1, 'normal': 2, 'squamous.cell.carcinoma': 3}
